<a href="https://colab.research.google.com/github/sirnaet/PS-Agents/blob/main/UCS_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq

kenya_highway_graph = {
    "Nairobi":  [("Naivasha", 88), ("Narok", 142)],
    "Naivasha": [("Nakuru", 68)],
    "Nakuru":   [("Kericho", 108), ("Eldoret", 158)],
    "Narok":    [("Bomet", 102)],
    "Bomet":    [("Kericho", 78), ("Kisii", 62)],
    "Kericho":  [("Kisumu", 76)],
    "Eldoret":  [("Kisumu", 117)],
    "Kisii":    [("Kisumu", 114)],
    "Kisumu":   [],
}

START, GOAL = "Nairobi", "Kisumu"

In [2]:
def ucs_with_trace(graph, start, goal, verbose=True):
    frontier = [(0, start)]
    best_cost = {start: 0}
    parent = {start: None}
    explored = set()
    step = 0

    def show_frontier():
        live = sorted((g, n) for g, n in frontier
                      if n not in explored and g == best_cost[n])
        return "[" + ", ".join(f"{n}({g})" for g, n in live) + "]"

    while frontier:
        if verbose:
            step += 1
            print(f"--- Iteration {step} ---")
            print(f"Frontier : {show_frontier()}")
            print(f"Explored : {sorted(explored) if explored else '{}'}")

        g, node = heapq.heappop(frontier)

        if node in explored or g > best_cost[node]:
            if verbose:
                print(f"Popped {node}({g}) is stale, skipping.\n")
            continue

        if verbose:
            print(f"Expand   : {node}  (g = {g} km)")

        # Goal test on expansion
        if node == goal:
            path, n = [], node
            while n is not None:
                path.append(n)
                n = parent[n]
            path.reverse()
            if verbose:
                print(f"GOAL TEST: {node} is the goal. Search ends.\n")
            return g, path, len(explored) + 1

        explored.add(node)
        if verbose:
            print("GOAL TEST: not the goal, generate children")

        for neighbor, dist in graph.get(node, []):
            new_g = g + dist
            if neighbor in explored:
                note = "already explored, discard"
            elif neighbor not in best_cost:
                best_cost[neighbor], parent[neighbor] = new_g, node
                heapq.heappush(frontier, (new_g, neighbor))
                note = "new, add to frontier"
            elif new_g < best_cost[neighbor]:
                note = f"cheaper than {best_cost[neighbor]}, update"
                best_cost[neighbor], parent[neighbor] = new_g, node
                heapq.heappush(frontier, (new_g, neighbor))
            else:
                note = f"not better than {best_cost[neighbor]}, discard"
            if verbose:
                print(f"   {node} -> {neighbor}: g = {g} + {dist} = {new_g}  ({note})")
        if verbose:
            print()

    return float("inf"), [], len(explored)

In [3]:
cost, path, expanded = ucs_with_trace(kenya_highway_graph, START, GOAL)

print("=" * 55)
print("UNIFORM COST SEARCH RESULT")
print("=" * 55)
print(f"Optimal Path   : {' -> '.join(path)}")
print(f"Total Distance : {cost} km")
print(f"Nodes expanded : {expanded}")

--- Iteration 1 ---
Frontier : [Nairobi(0)]
Explored : {}
Expand   : Nairobi  (g = 0 km)
GOAL TEST: not the goal, generate children
   Nairobi -> Naivasha: g = 0 + 88 = 88  (new, add to frontier)
   Nairobi -> Narok: g = 0 + 142 = 142  (new, add to frontier)

--- Iteration 2 ---
Frontier : [Naivasha(88), Narok(142)]
Explored : ['Nairobi']
Expand   : Naivasha  (g = 88 km)
GOAL TEST: not the goal, generate children
   Naivasha -> Nakuru: g = 88 + 68 = 156  (new, add to frontier)

--- Iteration 3 ---
Frontier : [Narok(142), Nakuru(156)]
Explored : ['Nairobi', 'Naivasha']
Expand   : Narok  (g = 142 km)
GOAL TEST: not the goal, generate children
   Narok -> Bomet: g = 142 + 102 = 244  (new, add to frontier)

--- Iteration 4 ---
Frontier : [Nakuru(156), Bomet(244)]
Explored : ['Nairobi', 'Naivasha', 'Narok']
Expand   : Nakuru  (g = 156 km)
GOAL TEST: not the goal, generate children
   Nakuru -> Kericho: g = 156 + 108 = 264  (new, add to frontier)
   Nakuru -> Eldoret: g = 156 + 158 = 314  (n